# MedScope AI — Colab Training and Weight Update

This notebook trains the YOLOv11 detector and Swin Transformer classifier, collects the best checkpoints, records a weight manifest, and optionally commits the updated weight metadata to GitHub.

> **Clinical safety:** This is research/development tooling, not a validated medical device. Do not use generated predictions or reports for patient care without appropriate clinical validation, regulatory review, security controls, and qualified human review.

In [ ]:
#@title 1. Runtime and configuration
import os, sys, json, platform, subprocess, glob, shutil, hashlib
from pathlib import Path

REPO_URL = "https://github.com/prey801/finalyearproject.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_DIR = "/content/finalyearproject" #@param {type:"string"}
YOLO_DATA = "data/raw/roboflow_malaria/data.yaml" #@param {type:"string"}
CLASSIFICATION_DATA = "data" #@param {type:"string"}
YOLO_EPOCHS = 100 #@param {type:"integer"}
YOLO_BATCH = 16 #@param {type:"integer"}
CLASS_EPOCHS = 20 #@param {type:"integer"}
CLASS_BATCH = 32 #@param {type:"integer"}
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT = "/content/drive/MyDrive/medscope-ai/weights" #@param {type:"string"}

try:
    import torch
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("PyTorch check failed:", e)
print("Training device:", DEVICE)

In [ ]:
#@title 2. Mount Google Drive (recommended for persistent checkpoints)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 3. Clone or update the repository
if not Path(PROJECT_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
else:
    %cd $PROJECT_DIR
    !git fetch origin
    !git checkout "$BRANCH"
    !git pull --ff-only origin "$BRANCH"
%cd $PROJECT_DIR
!git status --short
!find . -maxdepth 2 -type f | sort | head -200

In [ ]:
#@title 4. Install system and Python dependencies
!apt-get update -qq
!apt-get install -y -qq libgl1
!python -m pip install -U pip setuptools wheel
!python -m pip install "aiobotocore==2.13.0" "boto3==1.34.106" "botocore==1.34.106"
!python -m pip install -r backend/requirements.txt
!python -m pip install -r models/requirements.txt
!python -m pip install pytest qdrant-client ultralytics timm

import torch
print("CUDA after installation:", torch.cuda.is_available())

In [ ]:
#@title 4b. Download Roboflow Malaria Dataset
# Downloads the YOLO-format malaria blood smear dataset from Roboflow.
# Run this BEFORE cell 5 (dataset validation).
#
# Get your free API key: https://app.roboflow.com -> Settings -> API Keys
# WARNING: Do not commit a real API key to git.
# Store it safely as a Colab Secret (lock icon) named 'ROBOFLOW_API_KEY'.

ROBOFLOW_API_KEY = ""  #@param {type:"string"}
ROBOFLOW_WORKSPACE = "brian-musyoki-s-workspace"  #@param {type:"string"}
ROBOFLOW_PROJECT = "iml-malaria-td7v9"  #@param {type:"string"}
ROBOFLOW_VERSION = 1  #@param {type:"integer"}

import pathlib
from roboflow import Roboflow

if not ROBOFLOW_API_KEY:
    try:
        from google.colab import userdata
        ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
        print("Loaded ROBOFLOW_API_KEY from Colab Secrets.")
    except Exception:
        raise ValueError(
            "ROBOFLOW_API_KEY not set. Paste it into this cell or add it "
            "as a Colab Secret named 'ROBOFLOW_API_KEY'."
        )

dest = pathlib.Path(PROJECT_DIR) / "data/raw/roboflow_malaria"
if dest.exists() and dest.is_dir() and not any(dest.iterdir()):
    dest.rmdir()  # remove empty .gitkeep placeholder
dest.mkdir(parents=True, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8", location=str(dest), overwrite=True)

print(f"Dataset downloaded to: {dest}")
print("Contents:", sorted(f.name for f in dest.iterdir()))

if (dest / "data.yaml").exists():
    print("OK: data.yaml found -- ready for training.")
else:
    raise RuntimeError(
        "data.yaml missing after download. "
        "Check ROBOFLOW_PROJECT/ROBOFLOW_VERSION and ensure yolov8 export exists."
    )


In [ ]:
#@title 5. Validate datasets before training
from pathlib import Path
import yaml

root = Path(PROJECT_DIR)
yolo_yaml = root / YOLO_DATA
class_data = root / CLASSIFICATION_DATA

assert yolo_yaml.exists(), f"Missing YOLO dataset YAML: {yolo_yaml}. Upload/copy the Roboflow dataset first."
with open(yolo_yaml, "r") as f:
    ycfg = yaml.safe_load(f)
print("YOLO config:", ycfg)

assert class_data.exists(), f"Missing classification data directory: {class_data}"
images = [p for p in class_data.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}]
print(f"Classification images found: {len(images):,}")
if not images:
    raise RuntimeError("No classification images found. Prepare/split the NIH malaria dataset first.")

In [ ]:
#@title 6. Optional NIH dataset split
# Run this only when data/raw/nih_malaria contains the raw images and processed splits do not exist.
# !python -m scripts.split_data

In [ ]:
#@title 7. Train YOLOv11 detector
%cd $PROJECT_DIR

import os
# Use local SQLite for MLflow -- no server needed in Colab.
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
# Disable Ultralytics built-in MLflow callback to avoid duplicate logging.
os.environ["ULTRALYTICS_MLFLOW"] = "false"

!python -m scripts.train_yolo \\
  --data "$YOLO_DATA" \\
  --epochs "$YOLO_EPOCHS" \\
  --batch "$YOLO_BATCH" \\
  --device "$DEVICE"


In [ ]:
#@title 8. Train Swin Transformer classifier
%cd $PROJECT_DIR
!python -m scripts.train_classification   --data_dir "$CLASSIFICATION_DATA"   --epochs "$CLASS_EPOCHS"   --batch "$CLASS_BATCH"   --device "$DEVICE"

In [ ]:
#@title 9. Locate, verify, and persist trained weights
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil

root = Path(PROJECT_DIR)
weight_dir = root / "models" / "weights"
weight_dir.mkdir(parents=True, exist_ok=True)

candidates = []
for pattern in ["runs/**/weights/best.pt", "runs/**/best.pt", "outputs/**/best*.pt", "checkpoints/**/best*.pt", "**/best_model*.pth", "**/best*.ckpt"]:
    candidates.extend(root.glob(pattern))
# Exclude anything already copied into the destination.
candidates = sorted({p.resolve() for p in candidates if weight_dir.resolve() not in p.resolve().parents}, key=lambda p: p.stat().st_mtime)
assert candidates, "No trained checkpoints were found. Inspect the training logs and output paths."

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "weights": []}
for src in candidates:
    low = str(src).lower()
    if "detect" in low or "yolo" in low:
        dst_name = "yolov11_malaria_best.pt"
        model_name = "detection"
    elif "class" in low or "swin" in low:
        dst_name = "swin_malaria_best" + src.suffix
        model_name = "classification"
    else:
        dst_name = src.name
        model_name = "unclassified"
    dst = weight_dir / dst_name
    shutil.copy2(src, dst)
    item = {"model": model_name, "file": str(dst.relative_to(root)), "source": str(src.relative_to(root)),
            "bytes": dst.stat().st_size, "sha256": sha256(dst)}
    manifest["weights"].append(item)
    if USE_DRIVE:
        shutil.copy2(dst, Path(DRIVE_OUTPUT) / dst.name)
    print(item)

manifest_path = weight_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
if USE_DRIVE:
    shutil.copy2(manifest_path, Path(DRIVE_OUTPUT) / manifest_path.name)
print("Manifest:", manifest_path)

In [ ]:
#@title 10. Smoke-test the saved checkpoints
from ultralytics import YOLO
from pathlib import Path
import torch

root = Path(PROJECT_DIR)
yolo_path = root / "models/weights/yolov11_malaria_best.pt"
if yolo_path.exists():
    detector = YOLO(str(yolo_path))
    print("YOLO checkpoint loaded successfully:", yolo_path)

swin_files = list((root / "models/weights").glob("swin_malaria_best*"))
for p in swin_files:
    checkpoint = torch.load(p, map_location="cpu", weights_only=False)
    print("Swin checkpoint loaded successfully:", p, "type:", type(checkpoint).__name__)

In [ ]:
#@title 11. Point the backend to the new YOLO weight
import os
os.environ["YOLO_WEIGHTS_PATH"] = str(Path(PROJECT_DIR) / "models/weights/yolov11_malaria_best.pt")
print("YOLO_WEIGHTS_PATH=", os.environ["YOLO_WEIGHTS_PATH"])

# Optional API smoke test (runs until stopped):
# %cd $PROJECT_DIR
# !TESTING=true uvicorn backend.main:app --host 0.0.0.0 --port 8000

In [ ]:
#@title 12. Run model and API tests
%cd $PROJECT_DIR
!TESTING=true python -m pytest tests/test_primary_models.py -q

## Optional: commit metadata or weights to GitHub

Large model files should normally be stored with **Git LFS**, DVC, object storage, or a model registry—not regular Git. Never put a GitHub token directly in a notebook cell. Use Colab Secrets and restrict the token's permissions.

In [ ]:
#@title 13. Optional Git commit (does not push automatically)
%cd $PROJECT_DIR
!git status --short
# Recommended: commit the manifest/config only; track large weights with Git LFS or DVC.
# !git add models/weights/manifest.json
# !git commit -m "Update trained model weight manifest"
# !git push origin "$BRANCH"